In [ ]:
from load_data import load_data, TARGET

df = load_data()

print("Dataset shape:", df.shape)
print("Target column:", TARGET)

# Target and Outlier Analysis
**Author:** Diya Rajganesh  
**Deliverable 3 — Team 10**

## 1. Target selection

The target is `total_price` because our objective is to predict the full
ticket price paid by the customer, including taxes and fees. This is a
regression task because the target is numeric.

This analysis uses the team's synthetic flight-fare dataset. Its findings
must be checked again when the real dataset becomes available.

In [ ]:
import numpy as np

y = df[TARGET]

print("Target summary:")
print(y.describe())

print("\nMissing prices:", y.isna().sum())
print("Infinite prices:", np.isinf(y).sum())
print("Zero or negative prices:", (y <= 0).sum())
print("Skewness:", y.skew())

In [ ]:
print("Negative prices:", (y < 0).sum())
print("Zero prices:", (y == 0).sum())

display(
    df.loc[
        y <= 0,
        ["cabin", "origin", "destination",
         "base_fare", "taxes_fees", "total_price"]
    ].head(10)
)

In [ ]:
negative_rows = df.loc[df[TARGET] < 0]

sign_error = np.isclose(
    -negative_rows[TARGET],
    negative_rows["base_fare"] + negative_rows["taxes_fees"],
    atol=0.01,
    rtol=0
)

print("Negative prices checked:", len(negative_rows))
print("Match the negative of fare + fees:", sign_error.sum())

## 2. Target validity

The target has no missing or infinite values, but contains 30 negative
prices. All 30 equal the negative of base_fare + taxes_fees, within
0.01, suggesting sign errors.

These rows are excluded from this notebook's valid-price analysis.
The original data remains unchanged. The data-quality owner should
confirm whether to correct these values or exclude them from modeling.

In [ ]:
valid_df = df.loc[
    np.isfinite(df[TARGET]) & (df[TARGET] > 0)
].copy()

print("Rows used for valid-price analysis:", len(valid_df))
print("Median price:", valid_df[TARGET].median())
print("Mean price:", valid_df[TARGET].mean())
print("Skewness:", valid_df[TARGET].skew())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    valid_df[TARGET],
    bins=80,
    color="steelblue",
    edgecolor="white"
)

ax.axvline(
    valid_df[TARGET].mean(),
    color="red",
    linestyle="--",
    label="Mean"
)
ax.axvline(
    valid_df[TARGET].median(),
    color="orange",
    linestyle="--",
    label="Median"
)

ax.set_title("Total Price Distribution — Synthetic Data")
ax.set_xlabel("Total ticket price")
ax.set_ylabel("Number of observations")
ax.legend()

plt.tight_layout()
plt.show()

## 3. Target distribution

Among the 25,020 positive-price observations, the mean total price is
305.08 and the median is 236.99. Skewness is 4.32, and the histogram
shows a long right tail.

High prices require investigation rather than automatic removal.
Next, we compare cabins to see whether some of the overall price
variation reflects different ticket segments.

In [ ]:
cabin_summary = (
    valid_df.groupby("cabin", observed=True)[TARGET]
    .agg(["count", "mean", "median", "min", "max", "skew"])
    .sort_values("median")
)

display(cabin_summary.round(2))

In [ ]:
cabin_order = cabin_summary.index.tolist()

fig, ax = plt.subplots(figsize=(10, 5))

ax.boxplot(
    [
        valid_df.loc[valid_df["cabin"] == cabin, TARGET]
        for cabin in cabin_order
    ],
    tick_labels=cabin_order,
    showfliers=True
)

ax.set_title("Total Price by Cabin — Synthetic Data")
ax.set_xlabel("Cabin")
ax.set_ylabel("Total ticket price")

plt.tight_layout()
plt.show()

In [ ]:
prices = valid_df[TARGET]

q1 = prices.quantile(0.25)
q3 = prices.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

overall_outliers = (
    (prices < lower_bound) | (prices > upper_bound)
)

print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Lower boundary: {lower_bound:.2f}")
print(f"Upper boundary: {upper_bound:.2f}")
print(f"Flagged observations: {overall_outliers.sum():,}")
print(f"Percentage flagged: {overall_outliers.mean():.2%}")

In [ ]:
cabin_iqr = (
    valid_df.groupby("cabin", observed=True)[TARGET]
    .agg(
        count="count",
        Q1=lambda s: s.quantile(0.25),
        Q3=lambda s: s.quantile(0.75)
    )
)

cabin_iqr["IQR"] = cabin_iqr["Q3"] - cabin_iqr["Q1"]
cabin_iqr["lower"] = cabin_iqr["Q1"] - 1.5 * cabin_iqr["IQR"]
cabin_iqr["upper"] = cabin_iqr["Q3"] + 1.5 * cabin_iqr["IQR"]

# Give each observation the boundaries for its own cabin.
lower_by_row = valid_df["cabin"].map(cabin_iqr["lower"]).astype(float)
upper_by_row = valid_df["cabin"].map(cabin_iqr["upper"]).astype(float)

valid_df["cabin_iqr_flag"] = (
    (valid_df[TARGET] < lower_by_row)
    | (valid_df[TARGET] > upper_by_row)
)

cabin_iqr["flagged"] = (
    valid_df.groupby("cabin", observed=True)["cabin_iqr_flag"].sum()
)
cabin_iqr["flagged_pct"] = (
    100 * cabin_iqr["flagged"] / cabin_iqr["count"]
)

display(cabin_iqr.round(2))

In [ ]:
display(
    valid_df.nlargest(15, TARGET)[
        [
            "cabin",
            "origin",
            "destination",
            "distance_miles",
            "days_to_departure",
            "base_fare",
            "taxes_fees",
            "total_price",
            "cabin_iqr_flag"
        ]
    ]
)

In [ ]:
component_total = valid_df["base_fare"] + valid_df["taxes_fees"]

components_available = (
    valid_df["base_fare"].notna()
    & valid_df["taxes_fees"].notna()
)

matches_components = np.isclose(
    valid_df[TARGET], component_total,
    atol=0.01, rtol=0
)

matches_ten_times = np.isclose(
    valid_df[TARGET], 10 * component_total,
    atol=0.01, rtol=0
)

other_mismatch = (
    components_available
    & ~matches_components
    & ~matches_ten_times
)

print("Totals matching fare + fees:", matches_components.sum())
print("Totals matching 10 × (fare + fees):", matches_ten_times.sum())
print("Other mismatches:", other_mismatch.sum())
print("Rows missing fare components:", (~components_available).sum())

## 4. Outlier investigation and handling

The overall IQR rule flagged 1,644 positive-price observations (6.57%).
Cabin-specific boundaries flagged 591 (2.36%), showing that outlier
classification depends on the ticket segment.

Investigation found 30 positive totals equal to exactly 10 times
base_fare + taxes_fees, suggesting scaling errors. Previously,
30 negative totals were identified as likely sign errors.

Both groups are excluded from the analysis pending confirmation by
the data-quality owner. No raw values are overwritten.

Other high prices are retained for further analysis: an IQR flag
alone is insufficient evidence to cap or drop a valid fare.

In [ ]:
analysis_df = valid_df.loc[
    components_available & matches_components
].copy()

print("Rows after excluding suspected target errors:", len(analysis_df))
print(analysis_df[TARGET].describe().round(2))
print("Skewness:", round(analysis_df[TARGET].skew(), 3))

In [ ]:
# Overall boundaries after excluding suspected target errors.
prices = analysis_df[TARGET]
q1 = prices.quantile(0.25)
q3 = prices.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

analysis_df["overall_iqr_flag"] = (
    (prices < lower) | (prices > upper)
)

print(f"Overall boundaries: {lower:.2f} to {upper:.2f}")
print("Overall flagged:", analysis_df["overall_iqr_flag"].sum())

# Recalculate boundaries within each cabin.
groups = analysis_df.groupby("cabin", observed=True)[TARGET]

cabin_q1 = groups.transform(lambda s: s.quantile(0.25))
cabin_q3 = groups.transform(lambda s: s.quantile(0.75))
cabin_iqr_width = cabin_q3 - cabin_q1

analysis_df["cabin_iqr_flag"] = (
    (prices < cabin_q1 - 1.5 * cabin_iqr_width)
    | (prices > cabin_q3 + 1.5 * cabin_iqr_width)
)

summary = (
    analysis_df.groupby("cabin", observed=True)
    .agg(
        observations=(TARGET, "size"),
        flagged=("cabin_iqr_flag", "sum")
    )
)

summary["flagged_pct"] = (
    100 * summary["flagged"] / summary["observations"]
)

display(summary.round(2))

### Keep, cap, or drop?

I excluded 60 suspected target errors from this analysis: 30 negative
totals consistent with sign errors and 30 positive totals consistent
with a 10× scaling error. The source data was left unchanged.

Of the remaining 24,990 fares, the overall IQR rule flagged 1,621
(6.49%) and the within-cabin rule flagged 568 (2.27%). I recommend
keeping the remaining fares at their original values. An IQR flag
alone does not justify capping or dropping a fare. For example, the
highest remaining fare (3,294.05) matches its fare components and is
for a same-day Business/First ticket, making the high price plausible.

In [ ]:
prices = analysis_df[TARGET]
log_prices = np.log(prices)

print(f"Original price skewness: {prices.skew():.3f}")
print(f"Log price skewness: {log_prices.skew():.3f}")

## 5. Target transformation and loss recommendation

Taking the natural log of `total_price` reduced skewness from 3.055
to 0.186 in the synthetic data. I recommend starting with the
original `total_price` scale because the goal is to measure
prediction errors in the same currency units as the fares.

I recommend mean absolute error (MAE) as the initial training loss.
MAE gives each additional unit of error the same weight, while MSE
gives large misses disproportionately more weight. This is a reasonable
starting point given the remaining high-price tail. During validation,
report MAE overall and by cabin to see where predictions are least accurate.